### Análise da performance das features importances

In [1]:
import duckdb
import pandas as pd

### Lendo dataframe com tratamentos necessários

In [2]:
import pandas as pd

# Caminho completo para o CSV
caminho_arquivo = r'C:\Users\lucas\TG - ESG\Datasets\base_tratada.csv'

# Ler o arquivo
df_tratado = pd.read_csv(caminho_arquivo)

df_tratado.head()

,fulltimeemployees,auditrisk,boardrisk,compensationrisk,shareholderrightsrisk,overallrisk,payoutratio,beta,trailingpe,forwardpe,...,dividendyield,fiveyearavgdividendyield,earningsquarterlygrowth,lastdividendvalue,earningsgrowth,esg_risk_level,regularmarketdayrange_min,regularmarketdayrange_max,fiftytwoweekrange_min,fiftytwoweekrange_max
0,2781.0,6.0,7.0,9.0,10.0,9.0,0.0000,1.801,36.366970,10.830601,...,0.00,0.00,0.249626,0.00,0.226513,NaN,37.600,39.97,37.59,141.63
1,14000.0,7.0,8.0,7.0,6.0,7.0,0.4178,1.304,10.259872,9.257472,...,4.12,3.22,0.103000,0.83,0.130000,Medium,79.535,80.77,70.90,114.50
2,6400.0,5.0,5.0,8.0,8.0,7.0,0.3595,1.161,27.547590,27.235262,...,1.45,1.06,0.215000,1.74,0.301000,Medium,476.520,481.93,396.06,538.44
3,9600.0,8.0,10.0,8.0,7.0,9.0,0.0000,1.172,357.687500,25.895927,...,0.00,0.00,1.099000,0.00,1.250000,NaN,56.885,57.73,47.08,82.69
4,76000.0,8.0,2.0,3.0,3.0,1.0,0.0000,1.141,13.722940,12.261757,...,0.00,0.00,-0.320000,0.00,-0.245000,Medium,136.500,138.63,131.76,179.60


### Aplicando case when seguindo a feature importance e SHAP aplicados na RF

🔹 O que são quantis?

Quantis são pontos de corte que dividem uma distribuição de dados em partes proporcionais.

O quantil 0.20 (20%) é o valor abaixo do qual estão 20% das observações.

O quantil 0.80 (80%) é o valor abaixo do qual estão 80% das observações.

👉 Exemplos práticos:

Se você calcular o quantil 0.20 de totalDebt, vai encontrar o valor de dívida abaixo do qual estão as 20% empresas menos endividadas.

Já o quantil 0.80 te dá o valor de dívida abaixo do qual estão 80% das empresas, ou seja, só as 20% mais endividadas ficam acima dele.

🔹 Por que usar quantis no CASE WHEN?

Evita chute de limites arbitrários

Em vez de você decidir manualmente “dívida alta é acima de 1 bilhão”, você deixa o próprio dataset definir o que é “alto” ou “baixo”.

Isso é importante porque em ESG e finanças os números variam muito entre setores, países e períodos.

Normaliza para diferentes escalas

Se você usa dívida (totalDebt) ou liquidez (bidSize), os valores podem estar em milhões, bilhões ou até números muito pequenos.

Com quantis, o critério fica relativo à distribuição da sua amostra, não ao valor absoluto.

Alinha com a lógica do SHAP

O SHAP mostrou que variáveis como dívida alta, baixa liquidez, alta margem, etc. são importantes.

Mas ele não disse exatamente quanto é “alto” ou “baixo”.

Os quantis dão uma forma objetiva e reprodutível de traduzir isso em regra SQL.

🔹 Como funciona no seu exemplo

No código que montei, usei:

<= quantil 0.20 → sinal de baixo (ex.: dívida baixa, risco baixo, etc.).

>= quantil 0.80 → sinal de alto (ex.: dívida alta, margens altas, etc.).

Assim:

Classe Low → pega os 20% melhores sinais (baixo risco, margens altas, liquidez boa).

Classe High → pega os 20% piores sinais (alta dívida, baixa liquidez, etc.).

Classe Medium → o que sobra no meio (~60% das empresas).

Isso reflete exatamente a leitura dos seus gráficos SHAP:

extremos = mais risco ou mais sustentabilidade → fácil de classificar.

meio-termo = perfil intermediário → difícil, cai no “Medium”.

🔹 Resumindo

Quantis = pontos de corte baseados na distribuição dos dados.

Usar quantis garante que suas regras “alto/baixo” são coerentes com o dataset, sem chute.

É uma forma simples de transformar a interpretação SHAP em lógica SQL robusta.

### 🔍 Interpretação SHAP – Classe `High` (Risco ESG Elevado)

A análise SHAP mostra que o modelo associa as seguintes características a empresas com risco ESG alto:

- **Baixa liquidez** (`bidSize` baixo) e **alta dívida** (`totalDebt` alto) são os principais fatores que aumentam a previsão da classe `High`.
- Variáveis como `recommendationMean`, `freeCashflow`, `auditRisk` e `priceToBook` também contribuem significativamente para a predição.
- Empresas com **baixa confiança de analistas**, **fluxo de caixa fraco** e **estruturas financeiras frágeis** tendem a ser classificadas com maior risco ESG.
- Os resultados são coerentes com fundamentos ESG, especialmente no pilar **Governança** e na **sustentabilidade financeira**.

Essa interpretação sugere que o modelo capturou corretamente os sinais econômicos associados ao risco ESG alto.

### 🔍 Interpretação SHAP – Classe `Medium` (Risco ESG Intermediário)

A análise SHAP mostra que a classe `Medium` é definida por padrões **intermediários** nas variáveis:

- `recommendationMean`, `beta`, `operatingMargins`, `profitMargins` e `bidSize` aparecem como as features mais relevantes.
- Valores **intermediários** de volatilidade (`beta`), margens operacionais e recomendação de analistas aumentam a probabilidade de a empresa ser classificada como risco ESG médio.
- O modelo aprendeu a **distinguir a classe `Medium` como zona de equilíbrio**, onde empresas não se destacam nem positivamente (baixa probabilidade de risco) nem negativamente (alta probabilidade de risco).

📌 Esses resultados indicam coerência entre a construção da classe `Medium` e o comportamento esperado de empresas com perfil intermediário nos pilares ESG.

### 🔍 Interpretação SHAP – Classe `Low` (Risco ESG Baixo)

A explicação SHAP mostra que o modelo reconhece como sinais fortes de baixo risco ESG:

- **Baixo endividamento** (`totalDebt`) e **risco de auditoria reduzido** (`auditRisk`), diretamente ligados à boa governança.
- **Alta liquidez** (`bidSize`, `askSize`) e **participação institucional relevante** (`heldPercentInstitutions`), associados à confiança do mercado.
- **Margens operacionais e fluxo de caixa elevados**, que indicam saúde financeira e capacidade de sustentação de práticas ESG.

📌 O modelo parece capturar corretamente os **padrões esperados para empresas sustentáveis**, associando sinais de estabilidade e confiança com risco ESG baixo.

In [4]:
# Consultando com SQL direto no DataFrame
df = duckdb.query("""
WITH q AS (
  SELECT
    quantile_cont(totalDebt, 0.30) AS td_p30,
    quantile_cont(totalDebt, 0.70) AS td_p70,
    quantile_cont(bidSize,   0.30) AS bid_p30,
    quantile_cont(bidSize,   0.70) AS bid_p70,
    quantile_cont(auditRisk, 0.30) AS aud_p30,
    quantile_cont(auditRisk, 0.70) AS aud_p70,
    quantile_cont(profitMargins, 0.30) AS pm_p30,
    quantile_cont(profitMargins, 0.70) AS pm_p70,
    quantile_cont(operatingMargins, 0.30) AS opm_p30,
    quantile_cont(operatingMargins, 0.70) AS opm_p70,
    quantile_cont(recommendationMean, 0.30) AS rec_p30,
    quantile_cont(recommendationMean, 0.70) AS rec_p70
  FROM df_tratado
),

scored AS (
  SELECT
    f.*,
    (
      -- Pontos de risco (High)
      (CASE WHEN f.totalDebt >= q.td_p70 THEN 1 ELSE 0 END) +
      (CASE WHEN f.auditRisk >= q.aud_p70 THEN 1 ELSE 0 END) +
      (CASE WHEN f.bidSize <= q.bid_p30 THEN 1 ELSE 0 END) +
      (CASE WHEN f.recommendationMean >= q.rec_p70 THEN 1 ELSE 0 END) +
      (CASE WHEN f.profitMargins <= q.pm_p30 THEN 1 ELSE 0 END) +
      (CASE WHEN f.operatingMargins <= q.opm_p30 THEN 1 ELSE 0 END)
    ) AS risk_points
  FROM df_tratado f
  CROSS JOIN q
),

classified AS (
  SELECT
    *,
    CASE
      WHEN risk_points >= 4 THEN 'High'
      WHEN risk_points <= 1 THEN 'Low'
      ELSE 'Medium'
    END AS pred_rule
  FROM scored
)

SELECT * FROM classified;


""").to_df()

df.head()

,fulltimeemployees,auditrisk,boardrisk,compensationrisk,shareholderrightsrisk,overallrisk,payoutratio,beta,trailingpe,forwardpe,...,earningsquarterlygrowth,lastdividendvalue,earningsgrowth,esg_risk_level,regularmarketdayrange_min,regularmarketdayrange_max,fiftytwoweekrange_min,fiftytwoweekrange_max,risk_points,pred_rule
0,2781.0,6.0,7.0,9.0,10.0,9.0,0.0000,1.801,36.366970,10.830601,...,0.249626,0.00,0.226513,None,37.600,39.97,37.59,141.63,2,Medium
1,14000.0,7.0,8.0,7.0,6.0,7.0,0.4178,1.304,10.259872,9.257472,...,0.103000,0.83,0.130000,Medium,79.535,80.77,70.90,114.50,2,Medium
2,6400.0,5.0,5.0,8.0,8.0,7.0,0.3595,1.161,27.547590,27.235262,...,0.215000,1.74,0.301000,Medium,476.520,481.93,396.06,538.44,1,Low
3,9600.0,8.0,10.0,8.0,7.0,9.0,0.0000,1.172,357.687500,25.895927,...,1.099000,0.00,1.250000,None,56.885,57.73,47.08,82.69,3,Medium
4,76000.0,8.0,2.0,3.0,3.0,1.0,0.0000,1.141,13.722940,12.261757,...,-0.320000,0.00,-0.245000,Medium,136.500,138.63,131.76,179.60,4,High


In [5]:
import pandas as pd
from sklearn.metrics import classification_report

# 1) defina o conjunto canônico de rótulos
LABELS = ["Low", "Medium", "High"]

def _normalize_labels(s: pd.Series) -> pd.Series:
    s = s.astype(str).str.strip()             # remove espaços
    s = s.str.lower().map({"low":"Low","medium":"Medium","high":"High"}).fillna(s)  # normaliza caixa
    # mapeia números para rótulos, se houver
    num_map = { "0":"Low", "1":"Medium", "2":"High" }
    s = s.replace(num_map)
    # qualquer coisa fora de LABELS vira NaN
    s = s.where(s.isin(LABELS))
    return s

# 2) normaliza ambas as colunas
y_true = _normalize_labels(df["esg_risk_level"])
y_pred = _normalize_labels(df["pred_rule"])

# 3) remove linhas inválidas (NaN) para avaliação justa
mask_valid = y_true.notna() & y_pred.notna()
df_eval = df.loc[mask_valid].copy()
df_eval["y_true"] = y_true[mask_valid]
df_eval["y_pred"] = y_pred[mask_valid]

# 4) coluna de acerto/erro
df_eval["correct"] = (df_eval["y_true"] == df_eval["y_pred"]).astype(int)

# 5) acurácia (sem sklearn, evita erro de tipo)
accuracy = df_eval["correct"].mean()
print(f"Acurácia: {accuracy:.4f}")

# 6) matriz de confusão (completa, mesmo se faltar classe em y_pred)
cm = pd.crosstab(df_eval["y_true"], df_eval["y_pred"], 
                 rownames=["Actual"], colnames=["Predicted"], dropna=False)
# garante colunas/linhas em ordem
cm = cm.reindex(index=LABELS, columns=LABELS, fill_value=0)
print(cm)

# 7) métricas por classe (precisão/recall/F1)
# o classification_report aceita especificar as classes esperadas
print(classification_report(df_eval["y_true"], df_eval["y_pred"],
                            labels=LABELS, zero_division=0))


Acurácia: 0.4242
Predicted  Low  Medium  High
Actual                      
Low         74      95    24
Medium      57      99    28
High        14      29     9
              precision    recall  f1-score   support

         Low       0.51      0.38      0.44       193
      Medium       0.44      0.54      0.49       184
        High       0.15      0.17      0.16        52

    accuracy                           0.42       429
   macro avg       0.37      0.36      0.36       429
weighted avg       0.44      0.42      0.42       429



# ⚖️ Comparativo Geral — Modelos de Classificação de Risco ESG

## 📊 Tabela Comparativa de Desempenho

| Modelo         | Acurácia | Balanced Accuracy | Classe Melhor Prevista | Classe Crítica | Padrão Observado |
|----------------|----------|-------------------|------------------------|----------------|------------------|
| **KNN**        | ~0.53    | ~0.45             | Low (recall ~0.65)     | High (recall ~0.12) | Forte viés para `Medium`, desempenho global fraco. |
| **Random Forest** | ~0.59 | ~0.47             | Medium (f1 ~0.63)      | High (f1 ~0.19) | Predição mais estável, mas ainda com dificuldade em `High`. |
| **XGBoost**    | ~0.61    | ~0.50             | Low e Medium (f1 ~0.68 / 0.62) | High (f1 ~0.21) | Melhor equilíbrio entre classes, mas `High` segue sub-representada. |

---

## 🔍 Interpretação por Modelo

- **KNN**  
  - Modelo mais simples, interpretável mas pouco eficaz.  
  - Sofre com desequilíbrio da base → praticamente não identifica `High`.  
  - Pode servir apenas como baseline.  

- **Random Forest**  
  - Garante maior estabilidade e interpretabilidade.  
  - Predição razoável para `Medium` e `Low`.  
  - Ainda falha em capturar `High`, mas melhor que o KNN.  

- **XGBoost**  
  - Melhor desempenho geral (acurácia + balanced accuracy).  
  - Capacidade superior de discriminação entre `Low` e `Medium`.  
  - `High` segue crítica, mas o modelo é mais promissor para ajustes finos.  

---

## 📌 Conclusão Comparativa

- O **KNN não é adequado** para o problema: baixa acurácia e recall crítico para `High`.  
- O **Random Forest** oferece uma base sólida, mas ainda insuficiente para identificar riscos elevados.  
- O **XGBoost é o melhor candidato**: apresenta **maior acurácia e robustez setorial**, sendo o mais indicado para evolução do projeto.  

**Próximos Passos Recomendados:**
1. Aplicar **técnicas de balanceamento** (SMOTE, undersampling, oversampling).  
2. Ajustar **hiperparâmetros do XGBoost** (`scale_pos_weight`, `learning_rate`, `max_depth`).  
3. Incluir **variáveis ESG específicas** (ambientais, sociais, reputacionais) além das financeiras.  
4. Testar **ensembles híbridos** (e.g., *XGBoost + RF*) para aumentar recall da classe `High`.  

---

## 📚 Referências
- Cover, T., & Hart, P. (1967). *Nearest Neighbor Pattern Classification*. IEEE Transactions on Information Theory.  
- Breiman, L. (2001). *Random Forests*. Machine Learning.  
- Chen, T., & Guestrin, C. (2016). *XGBoost: A Scalable Tree Boosting System*. KDD.  
- Molnar, C. (2022). *Interpretable Machine Learning*.  
- Saito, T., & Rehmsmeier, M. (2015). *The precision-recall plot is more informative than the ROC plot when evaluating binary classifiers on imbalanced datasets*. PLoS ONE.  
